# 🌍 Sprint 1 — Task 1: Dual NO₂ + LST Visualization for CDMX

**Project:** *Dos Méxicos Bajo el Mismo Sol* — Urban heat islands, air quality, and environmental justice in CDMX.

**Notebook goal:** Load Sentinel-5P NO₂ (2023 annual) and Landsat 8 LST (summer 2023) over the CDMX state boundary, and visualize both as toggleable layers in a single `geemap` map.

**Architecture:** All reusable logic lives in the `src/` package (installed in editable mode). This notebook is a thin *pipeline orchestrator* — it imports the helpers and stitches them together.

---
### 🧠 Signal-processing cheat-sheet for this notebook

| GEE concept | Signal-processing analog |
|---|---|
| `ImageCollection` | Dataset of multichannel trials (each image = one trial) |
| `filterDate` / `filterBounds` | Epoch selection — keep only clean trials in the ROI |
| `filterMetadata('CLOUD_COVER', ...)` | Trial rejection by artifact threshold |
| `qa_value` mask (NO₂) | Confidence-weighted averaging (drop low-SNR epochs) |
| `QA_PIXEL` bitmask (Landsat) | Bitwise artifact flag (cloud, shadow) |
| `.median()` | ERP-style averaging across trials — robust to outliers |
| DN → K → °C scaling | Calibration (ADC counts → physical units) |

In [ ]:
# ---- Imports & Earth Engine initialization ----
# All reusable functions come from the installed `src` package.
# The reload block below forces Python to re-read the latest src/*.py
# from disk on every run, so editing a helper function and re-running
# this cell picks up the change without needing a kernel restart.
import importlib
import ee

import src
import src.aoi
import src.config
import src.landsat
import src.sentinel5p
import src.visualization
for mod in (src, src.aoi, src.config, src.landsat, src.sentinel5p, src.visualization):
    importlib.reload(mod)

from src import (
    CDMX_CENTER,
    EE_PROJECT_ID,
    build_dual_map,
    get_cdmx_aoi,
    load_lst_composite,
    load_no2_composite,
)

# Replace with your GCP project ID that has the Earth Engine API enabled.
# (The default placeholder is set in src/config.py.)
ee.Initialize(project=EE_PROJECT_ID)
print("✅ Earth Engine initialized")

In [ ]:
# ---- Load the CDMX AOI from the INEGI Marco Geoestadistico shapefile ----
# This returns a single-feature ee.FeatureCollection (CDMX state boundary).
aoi_cdmx = get_cdmx_aoi()

# Sanity check: print AOI area in km^2 (official CDMX area is ~1,485 km^2).
area_km2 = aoi_cdmx.geometry().area().divide(1e6).getInfo()
print(f"CDMX AOI area: {area_km2:,.1f} km²")

In [ ]:
# ---- Build the two composites ----
# LST: Landsat 8 Collection 2 L2, summer 2023 (June-August), cloud < 20%.
lst_summer_2023 = load_lst_composite(
    aoi_cdmx,
    start_date="2023-06-01",
    end_date="2023-09-01",
    cloud_max=20.0,
)
print("LST composite ready:", lst_summer_2023.bandNames().getInfo())

# NO₂: Sentinel-5P TROPOMI, full year 2023, cloud_fraction < 0.3.
no2_2023 = load_no2_composite(
    aoi_cdmx,
    start_date="2023-01-01",
    end_date="2023-12-31",
    cloud_max=0.3,
)
print("NO₂ composite ready:", no2_2023.bandNames().getInfo())

In [ ]:
# ---- Build the interactive dual-layer map ----
# Layers (top to bottom in the layer control):
#   1. CDMX AOI outline (white, 2 px stroke)
#   2. NO₂ (mol/m²) — 2023 annual median
#   3. LST (°C) — summer 2023 median (default visible)
Map = build_dual_map(lst_summer_2023, no2_2023, aoi_cdmx)
Map

### ✅ Checkpoint — what just happened

- **AOI:** CDMX state boundary loaded from `09ent.shp`, area printed (~1,485 km²).
- **LST composite:** cloud-masked median of Landsat 8 scenes from Jun–Aug 2023, scaled from `ST_B10` DN to °C, clipped to CDMX.
- **NO₂ composite:** median of Sentinel-5P overpasses across 2023 with `qa_value ≥ 0.75`, in mol/m².
- **Map:** toggleable layers with colorbars for both rasters and a white AOI outline.

### ⚡ MVP check

If the map renders, you're done with Sprint 1, Task 1. Don't worry about:
- Converting NO₂ to µg/m³ (needs ECMWF met bands — Sprint 2 work).
- Adjusting the colorbar ranges — the defaults are sane; tweak `LST_VIS_PARAMS` / `NO2_VIS_PARAMS` in `src/config.py` if the contrast feels off.
- Caching composites to `outputs/` — that comes after we have zonal statistics (Task 3).

### 🔜 Next step (Sprint 1, Task 2)

Add an **NDVI** layer to the same map. NDVI = `(B5 - B4) / (B5 + B4)`. It will be the third band in our environmental-justice composite. Two small additions:

1. New function `calculate_ndvi(image)` in `src/landsat.py`.
2. New function `load_ndvi_composite(...)` in `src/landsat.py`.
3. Extend `build_dual_map()` → `build_triple_map()` in `src/visualization.py`.

Want to move on to Task 2 now, or pause to inspect the map first?